In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from games.games.trivia_game import DEFAULT_QUESTION_BANK, QuestionsBank, TriviaGame, QuestionGenerator

/Users/kaischeikh/Desktop/Developer/projects/expression_game/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
generator = QuestionGenerator()

In [4]:
questions = generator.generate_questions(10, show_progress=True)

Generating questions: 100%|██████████| 10/10 [04:49<00:00, 29.00s/it]


In [5]:
import os
os.environ["QUESTIONS_DB_PATH"] = "./data/questions.db"

In [6]:
! mkdir ./data

mkdir: ./data: File exists


In [7]:
questions[-1].question

'What is the longest river in North America?'

In [8]:
generator.export_questions_to_db(questions)

10

In [15]:
import sqlite3
import pandas as pd
from pathlib import Path

In [16]:
root = Path(__file__).resolve().parents[1]

NameError: name '__file__' is not defined

In [19]:
connection = sqlite3.connect("./data/questions.db")

In [11]:
cursor = connection.cursor()

cursor.execute("""
SELECT name FROM sqlite_master
WHERE type='table'
""")

tables = cursor.fetchall()
print(tables)

[('questions',), ('sqlite_sequence',)]


In [12]:
query = """
SELECT
    *
FROM questions
"""

pd.read_sql_query(query, connection)["explanation"]


0     Christopher Nolan's 2010 film Inception center...
1     Water is composed of two hydrogen atoms bonded...
2     Baseball is a bat-and-ball game where players ...
3     Tom Seaver struck out 10 batters in Game 1 of ...
4     Bill Murray played the charismatic and sarcast...
5     The iconic phrase is first spoken in the 1977 ...
6     Pablo Picasso created Guernica in 1937 as a po...
7     Tokugawa Iemitsu, the third shogun of the Toku...
8     Pride and Prejudice is a novel by Jane Austen,...
9     The Simpsons has aired over 700 episodes, surp...
10    Uruguay hosted and won the inaugural FIFA Worl...
11    John Kennedy Toole’s novel was published after...
12    Faulkner's "The Sound and the Fury" uses multi...
13    The Last Starfighter featured a fully CGI char...
14    The Treaty of Paris signed in 1783 formally re...
15    'The Persistence of Memory', painted in 1931, ...
16    William Shakespeare is the author of the trage...
17    Salvador Dalí painted The Persistence of M

In [13]:
rows = connection.execute(
                """
                SELECT
                    category,
                    question,
                    options,
                    answer,
                    explanation,
                    difficulty
                FROM questions
                """
            ).fetchall()

In [14]:
row = rows[0]

In [15]:
import json

In [16]:
row

('Entertainment',
 'Which film directed by Christopher Nolan features a character named Dom Cobb?',
 '["Inception", "Memento", "Interstellar", "Dunkirk"]',
 'Inception',
 "Christopher Nolan's 2010 film Inception centers on Dom Cobb, a skilled extractor who steals secrets through dream‑sharing technology.",
 'medium')

In [ ]:
questions: list[Question] = []
for row in rows:
    options = json.loads(row["options"])
    questions.append(
        Question(
            category=row["category"],
            question=row["question"],
            options=tuple(options),
            answer=row["answer"],
            explanation=row["explanation"],
            difficulty=row["difficulty"],
        )
    )

In [3]:
bank = QuestionsBank()

In [4]:
bank.load_from_db()

20

In [ ]:
bank.bank[0]

False

In [10]:
import sqlite3

In [11]:
connection = sqlite3.connect("./data/questions.db")

In [12]:
connection.execute(
    """
    SELECT
        category,
        difficulty,
        SUM(CASE WHEN asked = 0 THEN 1 ELSE 0 END) AS remaining,
        COUNT(*) AS total
    FROM questions
    GROUP BY category, difficulty
    ORDER BY category, difficulty
    """
).fetchall()

[('Art', 'medium', 3, 3),
 ('Entertainment', 'easy', 2, 2),
 ('Entertainment', 'hard', 1, 1),
 ('Entertainment', 'medium', 3, 3),
 ('Geography', 'easy', 1, 1),
 ('History', 'hard', 1, 1),
 ('History', 'medium', 1, 1),
 ('Literature', 'easy', 2, 2),
 ('Literature', 'hard', 2, 2),
 ('Science', 'easy', 1, 1),
 ('Sports', 'easy', 2, 2),
 ('Sports', 'hard', 1, 1)]

In [ ]:
generator.export_questions_to_db()